# Binary Sentiment Analysis - Experimenting with Different Methods

There are a number of ways to classify text that involve different amounts of complexity. This notebook will experiment with different ways of conducting binary classification within a sentiment analysis task. We will use a very popular twitter sentiment analysis dataset for this. That dataset can be [found here on Kaggle.](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis?resource=download) This dataset comes with a pre-defined train/test split and there is a low proportion of data in the testing set, so we will add on another popular twitter sentiment analysis dataset to supplement that test set. That supplemental dataset can be [found here on Kaggle.](https://www.kaggle.com/datasets/yasserh/twitter-tweets-sentiment-dataset)

In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
train_df = pd.read_csv('twitter_training.csv', header=None, names=["tweet_id", "entity", "label", "tweet"]) 
val_df = pd.read_csv('twitter_validation.csv', header=None, names=["tweet_id", "entity", "label", "tweet"]) 

supplementary_df = pd.read_csv('Tweets.csv')[['text', 'sentiment']]
supplementary_df = supplementary_df.rename(columns={'text': 'tweet', 'sentiment': 'label'})

In [3]:
train_df.head()

,tweet_id,entity,label,tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [4]:
val_df.head()

,tweet_id,entity,label,tweet
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


We see each dataset contains 4 columns, we will only be using the tweet itself as well as the sentiment. Before working on model experiments, we will keep only the rows with a Positive or Negative label. 

In [5]:
train_df = train_df.loc[(train_df["label"] == "Positive") | (train_df["label"] == "Negative")][["tweet", "label"]]
val_df = val_df.loc[(val_df["label"] == "Positive") | (val_df["label"] == "Negative")][["tweet", "label"]]

supplementary_df = supplementary_df.loc[(supplementary_df["label"] == "positive") | (supplementary_df["label"] == "negative")]
supplementary_df = supplementary_df.replace({'label': {'positive': 'Positive', 'negative': 'Negative'}})
#Add the supplementary data to the validation set
val_df = pd.concat([val_df, supplementary_df], ignore_index=True)


Now that we have added the supplementary data to the validation set, there is now a solid amount of test data. 

In [6]:
train_df.shape

(43374, 2)

In [7]:
train_df["label"].value_counts()

label
Negative    22542
Positive    20832
Name: count, dtype: int64

In [8]:
val_df.shape

(16906, 2)

In [9]:
val_df["label"].value_counts()

label
Positive    8859
Negative    8047
Name: count, dtype: int64

We see the the classes are reasonably balanced, but for absolute fairness, we will downsample the training set to perfectly balance the classes in our training set as to not sway our models. 

In [10]:
min_count = train_df['label'].value_counts().min()

pos_df = train_df[train_df['label'] == 'Positive']
neg_df = train_df[train_df['label'] == 'Negative']
neg_df = neg_df.sample(n=min_count, random_state=42)
train_df = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=42).reset_index(drop=True)
train_df['label'].value_counts()

label
Positive    20832
Negative    20832
Name: count, dtype: int64

Lastly we will encode the pos/neg labels as numeric values and we will apply some text cleaning to standardize text. 

In [11]:
label_map = {'Positive': 1, 'Negative': 0}
train_df['label'] = train_df['label'].map(label_map).astype(int)
val_df['label'] = val_df['label'].map(label_map).astype(int)

In [12]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['tweet_clean'] = train_df['tweet'].apply(clean_tweet)
val_df['tweet_clean'] = val_df['tweet'].apply(clean_tweet)

## Method 1: Bag of Words with Logistic Regression
The most primitive way to turn our text into numeric features to use for prediction would be to use a "bag of words" approach where each word in the text becomes a column value and we count the number of times it appears in the sequence. 

In [ ]:
# Create Bag-of-Words features
vectorizer = CountVectorizer(stop_words='english', min_df=2)

X_train = vectorizer.fit_transform(train_df['tweet_clean'])
X_val = vectorizer.transform(val_df['tweet_clean'])
y_train = train_df['label']
y_val = val_df['label']

In [14]:
bow_model = LogisticRegression(max_iter=1000, random_state=42)
bow_model.fit(X_train, y_train)

y_pred = bow_model.predict(X_val)
print('Validation classification report:')
print(classification_report(y_val, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_val, y_pred))

Validation classification report:
              precision    recall  f1-score   support

           0       0.79      0.57      0.66      8047
           1       0.69      0.87      0.77      8859

    accuracy                           0.73     16906
   macro avg       0.74      0.72      0.72     16906
weighted avg       0.74      0.73      0.72     16906

Confusion matrix:
[[4586 3461]
 [1185 7674]]
